# Portuguese -> English NMT - Transformer (Kaggle, from scratch)

Self-contained rebuild of the Transformer. Train and infer in the **same** Kaggle
session, so there is no cross-version weight-loading problem.

### Setup
1. Upload `por.txt` (tab-separated `EN \t PT \t ...`) as a Kaggle Dataset and attach it.
   The loader auto-discovers any `por.txt` under `/kaggle/input/`.
2. Turn the **GPU (T4)** accelerator on.
3. Run all cells top to bottom.

### Quick smoke test first
Before the full run, set `CFG.MAX_SAMPLES = 20000` and `CFG.EPOCHS = 3` to confirm
everything works end to end (a few minutes). Then set `MAX_SAMPLES = None`,
`EPOCHS = 20` for the real training.


In [ ]:
import os, json, random, time, glob
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


class CFG:
    VOCAB_SIZE     = 10000
    MAX_LENGTH     = 40
    BATCH_SIZE     = 64
    D_MODEL        = 256
    NUM_LAYERS     = 4          # original was 3; 4 is a small, safe upgrade for full data
    NUM_HEADS      = 8
    DFF            = 1024
    DROPOUT        = 0.1
    EPOCHS         = 20
    WARMUP_STEPS   = 4000
    MAX_SAMPLES    = None       # None = full corpus; set e.g. 20000 for a quick test
    SEED           = 42
    WORK_DIR       = "/kaggle/working"
    TOKENIZER_PATH = "/kaggle/working/tokenizer.json"
    CKPT_DIR       = "/kaggle/working/checkpoints"


random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
tf.random.set_seed(CFG.SEED)
os.makedirs(CFG.CKPT_DIR, exist_ok=True)

## 1. Load and split the parallel corpus

In [ ]:
def find_data_file():
    cands = []
    for p in ["por.txt", "src/data/por.txt"]:
        if os.path.exists(p):
            cands.append(p)
    cands += sorted(glob.glob("/kaggle/input/**/por.txt", recursive=True))
    if not cands:
        cands += sorted(glob.glob("/kaggle/input/**/*.txt", recursive=True))
    if not cands:
        raise FileNotFoundError(
            "por.txt not found. Upload it as a Kaggle dataset and attach it.")
    return cands[0]


DATA_PATH = find_data_file()
print("Data file:", DATA_PATH)

pairs = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 2:
            continue
        en, pt = parts[0].strip(), parts[1].strip()
        if en and pt:
            pairs.append((pt, en))          # (source=Portuguese, target=English)

print("Total pairs:", len(pairs))
random.shuffle(pairs)
if CFG.MAX_SAMPLES:
    pairs = pairs[:CFG.MAX_SAMPLES]

n = len(pairs)
n_tr, n_val = int(0.8 * n), int(0.1 * n)
train_pairs = pairs[:n_tr]
val_pairs   = pairs[n_tr:n_tr + n_val]
test_pairs  = pairs[n_tr + n_val:]
print(f"Train {len(train_pairs)} | Val {len(val_pairs)} | Test {len(test_pairs)}")
for pt, en in train_pairs[:5]:
    print("PT:", pt, "||  EN:", en)

## 2. Train a shared Byte-Pair-Encoding tokenizer

Trained only on the **training** split. `[PAD]` is forced to id 0 (the model masks id 0).
A `ByteLevel` decoder is used so detokenization is clean (no stray `Ġ`).

In [ ]:
from tokenizers import (Tokenizer, models, trainers,
                        pre_tokenizers, processors, decoders)


def train_tokenizer(train_pairs, path, vocab_size):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[UNK]", "[START]", "[END]"],
    )
    texts = []
    for pt, en in train_pairs:
        texts.append(pt)
        texts.append(en)
    tok.train_from_iterator(texts, trainer=trainer)
    start_id, end_id = tok.token_to_id("[START]"), tok.token_to_id("[END]")
    tok.post_processor = processors.TemplateProcessing(
        single="[START] $A [END]",
        special_tokens=[("[START]", start_id), ("[END]", end_id)],
    )
    tok.save(path)
    return tok


if os.path.exists(CFG.TOKENIZER_PATH):
    tokenizer = Tokenizer.from_file(CFG.TOKENIZER_PATH)
    print("Loaded existing tokenizer.")
else:
    tokenizer = train_tokenizer(train_pairs, CFG.TOKENIZER_PATH, CFG.VOCAB_SIZE)
    print("Trained new tokenizer ->", CFG.TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID     = tokenizer.token_to_id("[PAD]")
START_ID   = tokenizer.token_to_id("[START]")
END_ID     = tokenizer.token_to_id("[END]")
assert PAD_ID == 0, "PAD must be id 0 (masking assumes 0)."
print(f"vocab={VOCAB_SIZE}  PAD={PAD_ID} START={START_ID} END={END_ID}")
print("sample ids:", tokenizer.encode("Eu gosto de futebol.").ids[:12])

## 3. tf.data input pipeline

`drop_remainder=True` on the training set keeps batch shapes static, which avoids
`tf.function` retracing.

In [ ]:
def encode_one(sentence):
    ids = tokenizer.encode(sentence).ids[:CFG.MAX_LENGTH]
    return ids + [PAD_ID] * (CFG.MAX_LENGTH - len(ids))


def build_arrays(pairs):
    enc, dec_in, dec_out = [], [], []
    for pt, en in pairs:
        e, d = encode_one(pt), encode_one(en)
        enc.append(e)
        dec_in.append(d[:-1])     # teacher forcing input
        dec_out.append(d[1:])     # shifted target
    return (np.asarray(enc, np.int32),
            np.asarray(dec_in, np.int32),
            np.asarray(dec_out, np.int32))


def make_ds(pairs, shuffle, drop_remainder):
    enc, dec_in, dec_out = build_arrays(pairs)
    ds = tf.data.Dataset.from_tensor_slices(
        ({"encoder_inputs": enc, "decoder_inputs": dec_in}, dec_out))
    if shuffle:
        ds = ds.shuffle(20000, seed=CFG.SEED, reshuffle_each_iteration=True)
    ds = ds.batch(CFG.BATCH_SIZE, drop_remainder=drop_remainder)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(train_pairs, shuffle=True,  drop_remainder=True)
val_ds   = make_ds(val_pairs,   shuffle=False, drop_remainder=False)
for x, y in train_ds.take(1):
    print("enc", x["encoder_inputs"].shape,
          "dec_in", x["decoder_inputs"].shape,
          "target", y.shape)

## 4. Model components

Note: custom layers deliberately **do not** name any `call` argument `mask`, and do
not enable `supports_masking`. All masking is passed explicitly. This is what keeps
weight loading robust across TensorFlow/Keras versions.

In [ ]:
def positional_encoding(length, depth):
    half = depth // 2
    positions = np.arange(length)[:, None]
    depths = np.arange(half)[None, :] / half
    angle = positions * (1.0 / (10000.0 ** depths))
    pos = np.concatenate([np.sin(angle), np.cos(angle)], axis=-1)
    return tf.cast(pos[None, ...], tf.float32)


class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, **kw):
        super().__init__(**kw)
        assert d_model % num_heads == 0
        self.d_model, self.num_heads = d_model, num_heads
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.out = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, b):
        x = tf.reshape(x, (b, -1, self.num_heads, self.depth))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, query, key, value, attn_mask=None):
        b = tf.shape(query)[0]
        q = self.split_heads(self.wq(query), b)
        k = self.split_heads(self.wk(key), b)
        v = self.split_heads(self.wv(value), b)
        logits = tf.matmul(q, k, transpose_b=True)
        logits = logits / tf.math.sqrt(tf.cast(self.depth, tf.float32))
        if attn_mask is not None:
            logits += attn_mask * -1e9
        weights = tf.nn.softmax(logits, axis=-1)
        ctx = tf.matmul(weights, v)
        ctx = tf.transpose(ctx, [0, 2, 1, 3])
        ctx = tf.reshape(ctx, (b, -1, self.d_model))
        return self.out(ctx), weights


def feed_forward(d_model, dff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation="relu"),
        tf.keras.layers.Dense(d_model),
    ])

In [ ]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1, **kw):
        super().__init__(**kw)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = feed_forward(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = tf.keras.layers.Dropout(rate)
        self.drop2 = tf.keras.layers.Dropout(rate)

    def call(self, x, training=False, padding_mask=None):
        a, _ = self.mha(x, x, x, attn_mask=padding_mask)
        o1 = self.ln1(x + self.drop1(a, training=training))
        f = self.ffn(o1)
        return self.ln2(o1 + self.drop2(f, training=training))


class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 vocab_size, max_len, rate=0.1, **kw):
        super().__init__(**kw)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos = positional_encoding(max_len, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, rate)
                           for _ in range(num_layers)]
        self.drop = tf.keras.layers.Dropout(rate)

    def call(self, x, training=False, padding_mask=None):
        L = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos[:, :L, :]
        x = self.drop(x, training=training)
        for layer in self.enc_layers:
            x = layer(x, training=training, padding_mask=padding_mask)
        return x

In [ ]:
def look_ahead_mask(size):
    return 1.0 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)


class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1, **kw):
        super().__init__(**kw)
        self.mha1 = MultiHeadAttention(d_model, num_heads)   # masked self-attn
        self.mha2 = MultiHeadAttention(d_model, num_heads)   # cross-attn
        self.ffn = feed_forward(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = tf.keras.layers.Dropout(rate)
        self.drop2 = tf.keras.layers.Dropout(rate)
        self.drop3 = tf.keras.layers.Dropout(rate)

    def call(self, x, enc_out, training=False,
             combined_mask=None, padding_mask=None):
        a1, _ = self.mha1(x, x, x, attn_mask=combined_mask)
        o1 = self.ln1(x + self.drop1(a1, training=training))
        a2, attn = self.mha2(o1, enc_out, enc_out, attn_mask=padding_mask)
        o2 = self.ln2(o1 + self.drop2(a2, training=training))
        f = self.ffn(o2)
        o3 = self.ln3(o2 + self.drop3(f, training=training))
        return o3, attn


class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 vocab_size, max_len, rate=0.1, **kw):
        super().__init__(**kw)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos = positional_encoding(max_len, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate)
                           for _ in range(num_layers)]
        self.drop = tf.keras.layers.Dropout(rate)

    def call(self, x, enc_out, training=False,
             combined_mask=None, padding_mask=None):
        L = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = x + self.pos[:, :L, :]
        x = self.drop(x, training=training)
        attn = None
        for layer in self.dec_layers:
            x, attn = layer(x, enc_out, training=training,
                            combined_mask=combined_mask, padding_mask=padding_mask)
        return x, attn

In [ ]:
class Transformer(tf.keras.Model):
    def __init__(self, vocab_size, num_layers, d_model, num_heads,
                 dff, max_len, rate=0.1, **kw):
        super().__init__(**kw)
        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                               vocab_size, max_len, rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                               vocab_size, max_len, rate)
        self.final_layer = tf.keras.layers.Dense(vocab_size)

    @staticmethod
    def padding_mask(seq):
        m = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return m[:, tf.newaxis, tf.newaxis, :]

    def encode(self, enc_in, training=False):
        pm = self.padding_mask(enc_in)
        return self.encoder(enc_in, training=training, padding_mask=pm), pm

    def decode(self, dec_in, enc_out, enc_pad_mask, training=False):
        L = tf.shape(dec_in)[1]
        combined = tf.maximum(look_ahead_mask(L), self.padding_mask(dec_in))
        dec_out, attn = self.decoder(dec_in, enc_out, training=training,
                                     combined_mask=combined,
                                     padding_mask=enc_pad_mask)
        return self.final_layer(dec_out), attn

    def call(self, inputs, training=False):
        enc_out, pm = self.encode(inputs["encoder_inputs"], training=training)
        logits, _ = self.decode(inputs["decoder_inputs"], enc_out, pm,
                                training=training)
        return logits

## 5. Loss, accuracy, learning-rate schedule

In [ ]:
_loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none")


def masked_loss(y_true, y_pred):
    mask = tf.cast(y_true != 0, tf.float32)
    loss = _loss_obj(y_true, y_pred) * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


def masked_accuracy(y_true, y_pred):
    pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    mask = tf.cast(y_true != 0, tf.float32)
    hits = tf.cast(tf.equal(y_true, pred), tf.float32) * mask
    return tf.reduce_sum(hits) / tf.reduce_sum(mask)


class WarmupSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup = float(warmup)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(
            tf.math.rsqrt(step), step * (self.warmup ** -1.5))

## 6. Build model and training steps

In [ ]:
model = Transformer(VOCAB_SIZE, CFG.NUM_LAYERS, CFG.D_MODEL, CFG.NUM_HEADS,
                    CFG.DFF, CFG.MAX_LENGTH, CFG.DROPOUT)
lr = WarmupSchedule(CFG.D_MODEL, CFG.WARMUP_STEPS)
optimizer = tf.keras.optimizers.Adam(lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

# build once so summary() works
_ = model({"encoder_inputs": tf.zeros((1, CFG.MAX_LENGTH), tf.int32),
           "decoder_inputs": tf.zeros((1, CFG.MAX_LENGTH - 1), tf.int32)})
model.summary()

train_loss_m = tf.keras.metrics.Mean()
train_acc_m  = tf.keras.metrics.Mean()


@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = masked_loss(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_loss_m(loss)
    train_acc_m(masked_accuracy(y, logits))


@tf.function
def val_step(x, y):
    logits = model(x, training=False)
    return masked_loss(y, logits), masked_accuracy(y, logits)

## 7. Train

Saves the best-on-validation weights to `/kaggle/working/checkpoints/`.

In [ ]:
best_val = float("inf")
best_path = os.path.join(CFG.CKPT_DIR, "transformer_best.weights.h5")
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(CFG.EPOCHS):
    t0 = time.time()
    train_loss_m.reset_state()
    train_acc_m.reset_state()
    for x, y in train_ds:
        train_step(x, y)

    vL, vA = tf.keras.metrics.Mean(), tf.keras.metrics.Mean()
    for x, y in val_ds:
        l, a = val_step(x, y)
        vL(l); vA(a)

    tl, ta = float(train_loss_m.result()), float(train_acc_m.result())
    vl, va = float(vL.result()), float(vA.result())
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl);   history["val_acc"].append(va)
    print(f"Epoch {epoch+1:2d}/{CFG.EPOCHS}  "
          f"loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f}  "
          f"({time.time()-t0:.0f}s)")

    if vl < best_val:
        best_val = vl
        model.save_weights(best_path)
        print("   saved best ->", best_path)

json.dump(history, open(os.path.join(CFG.CKPT_DIR, "history.json"), "w"), indent=2)
print("Done. Best val_loss:", round(best_val, 4))

## 8. Inference (greedy, encoder cached once per sentence)

In [ ]:
model.load_weights(best_path)


def encode_sentence(s):
    ids = tokenizer.encode(s).ids[:CFG.MAX_LENGTH]
    ids = ids + [PAD_ID] * (CFG.MAX_LENGTH - len(ids))
    return tf.constant([ids], tf.int32)


def translate(sentence, max_len=None):
    max_len = max_len or CFG.MAX_LENGTH
    enc_out, pm = model.encode(encode_sentence(sentence), training=False)
    out = [START_ID]
    for _ in range(max_len):
        logits, _ = model.decode(tf.constant([out], tf.int32), enc_out, pm,
                                 training=False)
        nxt = int(tf.argmax(logits[0, -1]))
        if nxt == END_ID:
            break
        out.append(nxt)
    return tokenizer.decode(out[1:], skip_special_tokens=True).strip()


tests = ["Eu gosto de futebol.", "Ela está estudando.", "Você fala inglês?",
         "Onde está meu carro?", "Eu não entendo.", "Você pode me ajudar?",
         "Nós vamos à escola.", "Ela gosta de música."]
for s in tests:
    print(f"PT: {s}\nEN: {translate(s)}\n" + "-" * 45)

## 9. Corpus BLEU on the held-out test set

Greedy decoding is slow (one forward per token), so this scores a subset. Increase
`N` for a tighter estimate.

In [ ]:
import subprocess, sys
try:
    import sacrebleu
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu

N = 500
sample = test_pairs[:N]
hyps = [translate(pt) for pt, _ in sample]
refs = [[en for _, en in sample]]
bleu = sacrebleu.corpus_bleu(hyps, refs)
print(f"Test BLEU over {len(sample)} sentences: {bleu.score:.2f}")
for (pt, en), h in list(zip(sample, hyps))[:8]:
    print(f"PT : {pt}\nREF: {en}\nHYP: {h}\n" + "-" * 45)

## 10. Save artifacts

Everything needed for local use is under `/kaggle/working/`:
- `checkpoints/transformer_best.weights.h5`
- `checkpoints/history.json`
- `tokenizer.json`

Download these from the Kaggle output panel. To reload locally, build the same
`Transformer`, run one dummy forward pass, then `model.load_weights(...)` — and use
the **same TensorFlow version Kaggle used** (printed in the first cell) to avoid the
cross-version loading issue.

In [ ]:
print("Artifacts in", CFG.WORK_DIR)
for root, _, files in os.walk(CFG.WORK_DIR):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"  {p}  ({os.path.getsize(p)/1e6:.2f} MB)")